In [1]:
import os
import pickle
import pandas as pd
from typing import Dict
from datetime import datetime

from azureml.core import Workspace, Datastore, Dataset

# Paths
EXPOSURE_PATH = "p1_postop/SSL/exposure/preop.csv"
OUTCOME_PATH = "p1_postop/SSL/targets/PRI.csv"
MAPPING_PATH = "Zahra/022026/Data/PreMEDS_MDPS/hash_to_integer_map.pkl"
        
#022026 for
DATASTORE_NAME = "researcher_data"

# Azure config
SUBSCRIPTION_ID = 'f8c5aac3-29fc-4387-858a-1f61722fb57a'
RESOURCE_GROUP = 'forskerpl-n0ybkr-rg'
WORKSPACE_NAME = 'forskerpl-n0ybkr-mlw'

# Columns
TIMESTAMP_COL = 'time'
PID_COL = 'subject_id'
ABSPOS_COL = 'abspos'


In [2]:
def get_workspace():
    return Workspace(
        subscription_id=SUBSCRIPTION_ID,
        resource_group=RESOURCE_GROUP,
        workspace_name=WORKSPACE_NAME
    )

ws = get_workspace()
print("Workspace loaded")

Workspace loaded


In [3]:
def print_dataset_summary(exposure_df, outcome_df, title="Dataset Summary"):
    print(f"\n===== {title} =====")
    
    print(f"Exposure data:")
    print(f"   - Records: {len(exposure_df):,}")
    print(f"   - Unique patients: {exposure_df[PID_COL].nunique():,}")
    
    print(f"\nOutcome data:")
    print(f"   - Records: {len(outcome_df):,}")
    print(f"   - Unique patients: {outcome_df[PID_COL].nunique():,}")

In [4]:
def read_csv_from_datastore(ws, datastore_name, path, rename_dict=None):
    datastore = Datastore.get(ws, datastore_name)
    dataset = Dataset.File.from_files((datastore, path))
    local_path = dataset.download("./temp", overwrite=True)[0]
    df = pd.read_csv(local_path)

    if rename_dict:
        df = df.rename(columns=rename_dict)

    return df

exposure = read_csv_from_datastore(
    ws, DATASTORE_NAME, EXPOSURE_PATH,
    rename_dict={"PID": PID_COL, "TIMESTAMP": TIMESTAMP_COL}
)

outcome = read_csv_from_datastore(
    ws, DATASTORE_NAME, OUTCOME_PATH,
    rename_dict={"PID": PID_COL, "TIMESTAMP": TIMESTAMP_COL}
)

print("Exposure rows:", len(exposure))
print("Outcome rows:", len(outcome))

Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


Resolving access token for scope "https://storage.azure.com/.default" using identity of type "MANAGED".
Getting data access token with Assigned Identity (client_id=clientid) and endpoint type based on configuration
{'infer_column_types': 'False', 'activity': 'download'}
{'infer_column_types': 'False', 'activity': 'download', 'activityApp': 'FileDataset'}
{'infer_column_types': 'False', 'activity': 'download'}
{'infer_column_types': 'False', 'activity': 'download', 'activityApp': 'FileDataset'}
Exposure rows: 767161
Outcome rows: 1813


In [5]:
def clean_data(df):
    df = df.dropna(subset=[TIMESTAMP_COL, PID_COL])
    df = df.drop_duplicates()
    return df

exposure = clean_data(exposure)
outcome = clean_data(outcome)

print("After cleaning:")
print("Exposure:", len(exposure))
print("Outcome:", len(outcome))

After cleaning:
Exposure: 767161
Outcome: 1813


In [6]:
print_dataset_summary(exposure, outcome, title="Before Filtering")


===== Before Filtering =====
Exposure data:
   - Records: 767,161
   - Unique patients: 526,364

Outcome data:
   - Records: 1,813
   - Unique patients: 1,788


In [7]:
def load_mapping(ws, datastore_name, path):
    datastore = Datastore.get(ws, datastore_name)
    dataset = Dataset.File.from_files((datastore, path))
    local_path = dataset.download("./temp", overwrite=True)[0]

    with open(local_path, "rb") as f:
        mapping = pickle.load(f)

    return mapping

pid_map = load_mapping(ws, DATASTORE_NAME, MAPPING_PATH)

print("Mapping loaded:", len(pid_map))

{'infer_column_types': 'False', 'activity': 'download'}
{'infer_column_types': 'False', 'activity': 'download', 'activityApp': 'FileDataset'}
Mapping loaded: 2218028


In [8]:
def map_pids(df, pid_map):
    df = df.copy()
    df[PID_COL] = df[PID_COL].astype(str).map(pid_map)
    df = df.dropna(subset=[PID_COL])
    df[PID_COL] = df[PID_COL].astype(int)
    return df

exposure = map_pids(exposure, pid_map)
outcome = map_pids(outcome, pid_map)

print("After mapping:")
print("Exposure unique patients:", exposure[PID_COL].nunique())
print("Outcome unique patients:", outcome[PID_COL].nunique())

After mapping:
Exposure unique patients: 526005
Outcome unique patients: 1788


In [9]:
def get_hours_since_epoch(series):
    ts = pd.to_datetime(series, utc=True, errors="coerce").dt.tz_localize(None)
    return (ts.view("int64") // 10**9) / 3600

exposure[ABSPOS_COL] = get_hours_since_epoch(exposure[TIMESTAMP_COL]).astype(int)
outcome[ABSPOS_COL] = get_hours_since_epoch(outcome[TIMESTAMP_COL]).astype(int)

print("abspos created")

abspos created


In [10]:
exposure_pids = set(exposure[PID_COL])
outcome_pids = set(outcome[PID_COL])

common_subjects = exposure_pids.intersection(outcome_pids)
missing_subjects = outcome_pids - exposure_pids

print("Unique exposure patients:", len(exposure_pids))
print("Unique outcome patients:", len(outcome_pids))
print("Common patients:", len(common_subjects))
print("Outcome patients WITHOUT exposure:", len(missing_subjects))

coverage = len(common_subjects) / len(outcome_pids)
print("Coverage:", coverage)

Unique exposure patients: 526005
Unique outcome patients: 1788
Common patients: 1788
Outcome patients WITHOUT exposure: 0
Coverage: 1.0


In [11]:
task_name = "PRI Prediction"

print(f"Task: {task_name}")         
print(f"   - Exposure data: {len(exposure):,} records from {exposure[PID_COL].nunique():,} unique patients")
print(f"   - Outcome data: {len(outcome):,} records from {outcome[PID_COL].nunique():,} unique patients")

Task: PRI Prediction
   - Exposure data: 766,764 records from 526,005 unique patients
   - Outcome data: 1,813 records from 1,788 unique patients


In [12]:
import os

# Local save directory
LOCAL_DIR = "./temp_data"
os.makedirs(LOCAL_DIR, exist_ok=True)

# Local file paths
EXPOSURE_LOCAL_PATH = os.path.join(LOCAL_DIR, "exposure_PRI_cv.csv")
OUTCOME_LOCAL_PATH = os.path.join(LOCAL_DIR, "outcome_PRI_cv.csv")

# Remote directory (ADLS)
REMOTE_DIR = "Zahra/062026/Corebehrt_CV/CreateOutcome/PRI"

In [13]:
#  اینجا حتماً از filtered data استفاده کن
exposure.to_csv(EXPOSURE_LOCAL_PATH, index=False)
outcome.to_csv(OUTCOME_LOCAL_PATH, index=False)

print("Files saved locally:")
print(EXPOSURE_LOCAL_PATH)
print(OUTCOME_LOCAL_PATH)

Files saved locally:
./temp_data/exposure_PRI_cv.csv
./temp_data/outcome_PRI_cv.csv


In [14]:
from azureml.core import Workspace, Datastore
from azure.storage.filedatalake import DataLakeServiceClient
from azure.identity import DefaultAzureCredential
import os

# ---------- Azure Config ----------
subscription_id = 'f8c5aac3-29fc-4387-858a-1f61722fb57a'
resource_group = 'forskerpl-n0ybkr-rg'
workspace_name = 'forskerpl-n0ybkr-mlw'
datastore_name = "researcher_data"

# ---------- Connect ----------
ws = Workspace(subscription_id, resource_group, workspace_name)
datastore = Datastore.get(ws, datastore_name)

account_name = datastore.account_name
filesystem_name = datastore.container_name

credential = DefaultAzureCredential()

# ---------- Upload Function ----------
def upload_to_adls2(local_file_path, remote_file_name, directory_path):
    
    # check file exists
    if not os.path.exists(local_file_path):
        raise FileNotFoundError(local_file_path)

    service_client = DataLakeServiceClient(
        account_url=f"https://{account_name}.dfs.core.windows.net",
        credential=credential
    )

    file_system_client = service_client.get_file_system_client(filesystem_name)

    # create directory if not exists
    directory_client = file_system_client.get_directory_client(directory_path)
    try:
        directory_client.create_directory()
    except:
        pass

    file_client = directory_client.get_file_client(remote_file_name)

    print(f"Uploading {remote_file_name} ...")

    with open(local_file_path, "rb") as f:
        file_client.upload_data(f, overwrite=True)

    print(f" Uploaded → {directory_path}/{remote_file_name}")


# ---------- Paths ----------
directory_path = "Zahra/062026/Corebehrt_CV/CreateOutcome/PRI"

# ---------- Upload ----------
upload_to_adls2(
    "./temp_data/exposure_PRI_cv.csv",
    "exposure.csv",
    directory_path
)

upload_to_adls2(
    "./temp_data/outcome_PRI_cv.csv",
    "outcome.csv",
    directory_path
)

Uploading exposure.csv ...
 Uploaded → Zahra/062026/Corebehrt_CV/CreateOutcome/PRI/exposure.csv
Uploading outcome.csv ...
 Uploaded → Zahra/062026/Corebehrt_CV/CreateOutcome/PRI/outcome.csv
